# 03 — Feature EngineeringThis notebook builds every derived column used across the dashboard:| New Column | Purpose ||---|---|| `salary_usd` | A single comparable annual salary figure || `experience_category` | Entry / Mid / Senior / Executive Level, from the job title || `work_type` | On-site / Hybrid / Remote, from `is_remote` + description keywords || `company_size_clean` | Small / Medium / Large, estimated from description text || `benefits_score` | A 0–10 score based on how many benefit keywords appear in the description |

In [1]:
import pandas as pdimport numpy as npfrom pathlib import PathBASE_DIR = Path.cwd()df = pd.read_csv(BASE_DIR / "data" / "clean" / "uk_jobs_clean.csv")print("Rows loaded:", len(df))

Rows loaded: 726

### Feature 1 — `salary_usd`: normalising every salary to one comparable figure

In [2]:
GBP_TO_USD = 1.27def to_annual_gbp(row):    if pd.isna(row["salary_min"]) or pd.isna(row["salary_max"]):        return np.nan    mid = (row["salary_min"] + row["salary_max"]) / 2    period = str(row["salary_period"]).upper()    if period == "HOUR":        return mid * 40 * 52    elif period == "DAY":        return mid * 5 * 52    elif period == "WEEK":        return mid * 52    elif period == "MONTH":        return mid * 12    else:        return middf["salary_annual_gbp"] = df.apply(to_annual_gbp, axis=1)df["salary_usd"] = df["salary_annual_gbp"] * GBP_TO_USDdf[["salary_min", "salary_max", "salary_period", "salary_usd"]].dropna().head()

   salary_min  salary_max salary_period    salary_usd0     45000.0     65000.0          YEAR  69850.0000001     35000.0     50000.0          YEAR  53975.0000002     90000.0    140000.0          YEAR 146050.0000003        25.0        35.0          HOUR  79248.0000004     55000.0     75000.0          YEAR  82550.000000

### Feature 2 — `experience_category`: reading seniority from the job title

In [3]:
def categorise_experience(title):    title = str(title).lower()    if any(k in title for k in ["junior", "graduate", "entry", "jr", "intern"]):        return "Entry Level"    elif any(k in title for k in ["senior", "sr.", "sr ", "lead", "principal"]):        return "Senior Level"    elif any(k in title for k in ["manager", "director", "vp", "chief", "head of", "executive"]):        return "Executive Level"    else:        return "Mid Level"df["experience_category"] = df["job_title"].apply(categorise_experience)df["experience_category"].value_counts()

experience_categoryMid Level          343Senior Level        255Entry Level         93Executive Level      35Name: count, dtype: int64

### Feature 3 — `work_type`: On-site / Hybrid / Remote

In [4]:
def categorise_work_type(row):    desc = str(row["job_description"]).lower()    is_remote = row["is_remote"]    if is_remote == True:        if "hybrid" in desc:            return "Hybrid"        return "Remote"    else:        if "hybrid" in desc or "remote" in desc:            return "Hybrid"        return "On-site"df["work_type"] = df.apply(categorise_work_type, axis=1)df["work_type"].value_counts()

work_typeOn-site    260Hybrid     255Remote     214Name: count, dtype: int64

### Feature 4 — `company_size_clean`: estimating company size from the description

In [5]:
# JSearch does not return a clean employee-count field, so we estimate company size# from common phrases used in job descriptions (a simple keyword heuristic).def categorise_company_size(desc):    desc = str(desc).lower()    large_signals  = ["fortune 500", "multinational", "global leader", "thousands of employees",                       "10,000+", "enterprise"]    small_signals  = ["startup", "small team", "early-stage", "seed-funded", "small but"]    if any(s in desc for s in large_signals):        return "Large"    elif any(s in desc for s in small_signals):        return "Small"    else:        return "Medium"df["company_size_clean"] = df["job_description"].apply(categorise_company_size)df["company_size_clean"].value_counts()

company_size_cleanMedium    441Large     201Small      84Name: count, dtype: int64

### Feature 5 — `benefits_score`: scoring how many benefits a posting advertises (0–10)

In [6]:
benefit_keywords = [    "pension", "health insurance", "private medical", "bonus", "equity",    "stock options", "dental", "gym membership", "life insurance",    "flexible hours", "remote work", "25 days holiday", "wellbeing",    "training budget", "learning budget", "season ticket loan"]def score_benefits(desc):    desc = str(desc).lower()    hits = sum(1 for kw in benefit_keywords if kw in desc)    score = round((hits / len(benefit_keywords)) * 10, 1)    return min(score, 10.0)df["benefits_score"] = df["job_description"].apply(score_benefits)print("Average benefits score:", round(df["benefits_score"].mean(), 2))df["benefits_score"].describe()

Average benefits score: 7.44count    726.000000mean       7.44std        1.62min        2.5025%        6.3050%        7.5075%        8.80max       10.00Name: benefits_score, dtype: float64

In [7]:
output_path = BASE_DIR / "data" / "engineered" / "uk_jobs_features.csv"output_path.parent.mkdir(parents=True, exist_ok=True)df.to_csv(output_path, index=False)print("FEATURE-ENGINEERED DATA SAVED")print("Final row count:", len(df))print("Columns added: salary_usd, experience_category, work_type, company_size_clean, benefits_score")

FEATURE-ENGINEERED DATA SAVED\nFinal row count: 726\nColumns added: salary_usd, experience_category, work_type, company_size_clean, benefits_score

**Output of this notebook:** `data/engineered/uk_jobs_features.csv` — the fully feature-engineered dataset, including `salary_usd`, `experience_category`, `work_type`, `company_size_clean`, and `benefits_score`. This is the file loaded into PostgreSQL in `04_sql_export.ipynb`.**Note on methodology:** `company_size_clean` and `benefits_score` are derived using keyword-based heuristics on the job description text, since the JSearch API does not return a structured employee-count or benefits field. This is a simple, explainable approach — a natural next step would be replacing it with an NLP-based classifier for higher accuracy.